# 01 - Generacion de Datos Sinteticos

## Objetivo
Generar datos sinteticos que simulan una red de fibra optica con:
- **200 segmentos** de fibra optica
- **7 dias** de lecturas cada **15 minutos**
- **~130,000 registros** totales
- **3 tipos de falla** con firmas pre-falla detectables

## Por que datos sinteticos?
Los datos reales de redes de fibra optica son confidenciales. Generamos datos
sinteticos que replican las distribuciones y patrones estadisticos reales,
permitiendo demostrar la metodologia sin necesidad de datos propietarios.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generator import FiberNetworkGenerator
from src.utils.config_loader import load_config

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

config = load_config()
print(f"Segmentos: {config['data_generation']['n_segments']}")
print(f"Dias: {config['data_generation']['days']}")
print(f"Intervalo: {config['data_generation']['reading_interval_min']} min")

In [ ]:
# Generar datos
generator = FiberNetworkGenerator()
df, path = generator.generate_and_save()

print(f"\nShape: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print(f"Archivo: {path}")
df.head(10)

In [ ]:
# Resumen estadistico
print("=" * 60)
print("RESUMEN ESTADISTICO")
print("=" * 60)
print(f"\nTotal registros: {len(df):,}")
print(f"Segmentos unicos: {df['segment_id'].nunique()}")
print(f"Rango temporal: {df['timestamp'].min()} a {df['timestamp'].max()}")
print(f"\nDistribucion de fallas:")
print(df['fault_type'].value_counts())
print(f"\n% con falla: {df['is_fault'].mean()*100:.1f}%")
df.describe()

In [ ]:
# Visualizar un segmento con falla
faulty_segments = df[df['is_fault'] == 1]['segment_id'].unique()
sample_seg = faulty_segments[0]
seg_data = df[df['segment_id'] == sample_seg].copy()

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle(f'Segmento {sample_seg} - Tipo de falla: {seg_data[seg_data["is_fault"]==1]["fault_type"].iloc[0]}',
             fontsize=14, fontweight='bold')

metrics = ['optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db', 'chromatic_dispersion', 'temperature']
titles = ['Potencia Optica (dBm)', 'Atenuacion (dB/km)', 'BER', 'OSNR (dB)', 'Dispersion Cromatica', 'Temperatura (C)']

for ax, metric, title in zip(axes.flat, metrics, titles):
    ax.plot(seg_data['timestamp'], seg_data[metric], linewidth=0.8, alpha=0.8)
    
    # Marcar zona de falla en rojo
    fault_mask = seg_data['is_fault'] == 1
    if fault_mask.any():
        fault_start = seg_data.loc[fault_mask, 'timestamp'].iloc[0]
        ax.axvline(x=fault_start, color='red', linestyle='--', alpha=0.7, label='Inicio falla')
    
    # Marcar zona pre-falla en amarillo
    pre_fault_mask = seg_data['pre_fault_hours'] > 0
    if pre_fault_mask.any():
        pre_start = seg_data.loc[pre_fault_mask, 'timestamp'].iloc[0]
        ax.axvline(x=pre_start, color='orange', linestyle='--', alpha=0.7, label='Inicio pre-falla')
    
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Distribucion de fallas por tipo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart de tipos de falla
fault_counts = df[df['is_fault'] == 1]['fault_type'].value_counts()
axes[0].pie(fault_counts.values, labels=fault_counts.index, autopct='%1.1f%%',
            colors=['#ff6b6b', '#ffa726', '#66bb6a'])
axes[0].set_title('Distribucion de Tipos de Falla')

# Histograma de horas pre-falla
pre_fault_data = df[df['pre_fault_hours'] > 0]['pre_fault_hours']
axes[1].hist(pre_fault_data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Horas antes de la falla')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion de Ventana Pre-Falla')

plt.tight_layout()
plt.show()

print(f"\nSegmentos con falla: {len(faulty_segments)} de {df['segment_id'].nunique()}")
print(f"Registros normales: {(df['fault_type']=='normal').sum():,}")
print(f"Registros con falla: {df['is_fault'].sum():,}")